In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Estructura del set de datos

- Meses: 12
- Viajes: 4,497,877
- Estaciones: 471

| Variable | Tipo de dato | Descripción |
|---|---|---|
| Inicio_del_viaje | `datetime64[us]` | Fecha y hora en que inicia el viaje. |
| Fin_del_viaje | `datetime64[us]` | Fecha y hora en que termina el viaje. |
| Origen_Id | `int64` | Identificador de la estación donde inicia el viaje. |
| Destino_Id | `int64` | Identificador de la estación donde termina el viaje. |
| latitude | `float64` | Latitud de la estación de origen. |
| longitude | `float64` | Longitud de la estación de origen. |
| dpcapacity | `float64` | Capacidad de anclajes registrada para la estación de origen. |

In [2]:
df = pd.read_parquet(r"C:\Users\PC\Desktop\TC5061-Proyecto-Equipo12\mi-bici-experiment\data\interim\trips_10_2024_to_09_2026.parquet")

df

,inicio_del_viaje,fin_del_viaje,origen_id,destino_id,longitude,latitude,dpcapacity
0,2024-10-01,2024-10-01,60,359,-103.353539,20.672642,11
1,2024-10-01,2024-10-01,16,79,-103.365929,20.682144,19
2,2024-10-01,2024-10-01,16,79,-103.365929,20.682144,19
3,2024-10-01,2024-10-01,49,51,-103.362350,20.674931,15
4,2024-10-01,2024-10-01,66,86,-103.370567,20.673321,19
...,...,...,...,...,...,...,...
9041960,2026-09-30,2026-09-30,21,254,-103.368057,20.678814,23
9041961,2026-09-30,2026-09-30,17,17,-103.369347,20.683191,15
9041962,2026-09-30,2026-09-30,21,33,-103.368057,20.678814,23
9041963,2026-09-30,2026-09-30,17,15,-103.369347,20.683191,15


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9041965 entries, 0 to 9041964
Data columns (total 7 columns):
 #   Column            Dtype         
---  ------            -----         
 0   inicio_del_viaje  datetime64[us]
 1   fin_del_viaje     datetime64[us]
 2   origen_id         int16         
 3   destino_id        int16         
 4   longitude         float32       
 5   latitude          float32       
 6   dpcapacity        int8          
dtypes: datetime64[us](2), float32(2), int16(2), int8(1)
memory usage: 250.1 MB


In [4]:
df[df["longitude"].isnull()]

,inicio_del_viaje,fin_del_viaje,origen_id,destino_id,longitude,latitude,dpcapacity


In [5]:
df.isnull().sum()

inicio_del_viaje    0
fin_del_viaje       0
origen_id           0
destino_id          0
longitude           0
latitude            0
dpcapacity          0
dtype: int64

## Se revisa el número de estaciones únicas que contiene el set de datos.

In [3]:
estaciones_unicas = pd.Index(df["Origen_Id"].dropna().unique()).union(
    df["Destino_Id"].dropna().unique()
).sort_values()

print(f"Total de estaciones únicas (origen y destino): {len(estaciones_unicas)}")

Total de estaciones únicas (origen y destino): 472


# Se optimiza el uso de memoria del set de datos.

In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4497877 entries, 0 to 4497876
Data columns (total 7 columns):
 #   Column            Dtype         
---  ------            -----         
 0   Inicio_del_viaje  datetime64[us]
 1   Fin_del_viaje     datetime64[us]
 2   Origen_Id         int64         
 3   Destino_Id        int64         
 4   latitude          float64       
 5   longitude         float64       
 6   dpcapacity        float64       
dtypes: datetime64[us](2), float64(3), int64(2)
memory usage: 240.2 MB


In [24]:
for col in ["Origen_Id", "Destino_Id"]:
    df[col] = pd.to_numeric(df[col], downcast="integer")

for col in ["latitude", "longitude", "dpcapacity"]:
    df[col] = pd.to_numeric(df[col], downcast="float")

In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4497877 entries, 0 to 4497876
Data columns (total 7 columns):
 #   Column            Dtype         
---  ------            -----         
 0   Inicio_del_viaje  datetime64[us]
 1   Fin_del_viaje     datetime64[us]
 2   Origen_Id         int16         
 3   Destino_Id        int16         
 4   latitude          float32       
 5   longitude         float32       
 6   dpcapacity        float32       
dtypes: datetime64[us](2), float32(3), int16(2)
memory usage: 137.3 MB


# Análisis descriptivo 

In [27]:
num_vars = df.select_dtypes(include="number")

num_vars.describe().T

,count,mean,std,min,25%,50%,75%,max
Origen_Id,4497877.0,163.609016,121.318466,2.000000,52.000000,159.000000,265.000000,519.000000
Destino_Id,4497877.0,165.456922,123.592169,2.000000,51.000000,159.000000,268.000000,519.000000
latitude,4495917.0,20.679502,0.016470,20.636129,20.670511,20.676067,20.685040,20.739523
longitude,4495917.0,-103.360283,0.019514,-103.433212,-103.371788,-103.357681,-103.348061,-103.297112
dpcapacity,4495917.0,18.278328,5.441606,11.000000,15.000000,19.000000,19.000000,39.000000


# Análisis de la serie de tiempo

## 1 año de datos (Octubre 2025 - Septiembre 2026)

In [4]:
viajes_por_dia = (
    df.groupby(df["Inicio_del_viaje"].dt.normalize(), sort=True)
    .size()
    .asfreq("D", fill_value=0)
)
promedio_movil_7d = viajes_por_dia.rolling(window=7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(15, 5))
viajes_por_dia.plot(ax=ax, linewidth=0.8, alpha=0.55, label="Viajes por día")
promedio_movil_7d.plot(ax=ax, linewidth=2, label="Promedio móvil (7 días)")
ax.set_title("Viajes iniciados por día")
ax.set_xlabel("Fecha")
ax.set_ylabel("Número de viajes")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

AttributeError: Can only use .dt accessor with datetimelike values

## 3 años de datos (Enero 2024 - Septiembre 2026)

![Serie de tiempo de viajes de tres años](images/serie_de_tiempo_3_años.png)

## 5 años de datos (Enero 2022 - Septiembre 2026)

![Serie de tiempo de viajes de cinco años](images/serie_de_tiempo_5_años.png)

## Viajes promedio por día

In [30]:
dias_semana = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

promedio_por_dia_semana = (
    viajes_por_dia.groupby(viajes_por_dia.index.dayofweek)
    .mean()
    .reindex(range(7))
)
promedio_por_dia_semana.index = dias_semana
promedio_por_dia_semana = promedio_por_dia_semana.rename("Promedio de viajes")
promedio_por_dia_semana.to_frame().round(2)

,Promedio de viajes
Lunes,12861.13
Martes,14023.50
Miércoles,13890.43
Jueves,13509.48
Viernes,13250.52
Sábado,9943.79
Domingo,8751.65


## Viajes promedio por mes

In [31]:
promedio_diario_por_mes = (
    viajes_por_dia.resample("MS").mean()
    .rename("Promedio diario de viajes")
    .to_frame()
    .round(2)
)
promedio_diario_por_mes.index = promedio_diario_por_mes.index.strftime("%Y-%m")
promedio_diario_por_mes.index.name = "Mes"
promedio_diario_por_mes

,Promedio diario de viajes
Mes,
2025-10,13550.45
2025-11,12677.60
2025-12,10820.00
2026-01,11521.94
2026-02,11919.86
2026-03,12136.87
2026-04,11728.37
2026-05,10190.16
2026-06,11843.43


## Número de estaciones unicas por mes

In [32]:
estaciones_unicas_por_mes = (
    df.groupby(df["Inicio_del_viaje"].dt.to_period("M"), sort=True)
    .apply(
        lambda mes: len(
            pd.Index(mes["Origen_Id"].dropna().unique()).union(
                mes["Destino_Id"].dropna().unique()
            )
        )
    )
    .rename("Estaciones únicas")
    .to_frame()
)
estaciones_unicas_por_mes.index.name = "Mes"
estaciones_unicas_por_mes.index = estaciones_unicas_por_mes.index.astype(str)
estaciones_unicas_por_mes

,Estaciones únicas
Mes,
2025-10,368
2025-11,367
2025-12,368
2026-01,368
2026-02,368
2026-03,369
2026-04,368
2026-05,371
2026-06,427


# Hallazgos

## 1. Patrones temporales

Al analizar las gráficas de las series de tiempo, se pueden identificar patrones temporales bien definidos y recurrentes en el uso del sistema de bicicletas MiBici.

Los principales patrones observados son:

- **Comportamiento semanal:** La mayoría de los viajes se realiza de lunes a viernes, mientras que los sábados y domingos se registra una disminución en el número de viajes.
- **Disminución durante diciembre y enero:** El número de viajes baja considerablemente entre finales de diciembre e inicios de enero.
- **Disminución durante marzo y abril:** Se observa otra caída importante en el número de viajes durante este periodo.

## 2. Interpretación de los patrones

Los patrones identificados sugieren que el sistema MiBici podría utilizarse principalmente como medio de transporte para desplazarse hacia los lugares de trabajo o estudio.

Esta hipótesis se apoya en la disminución de viajes durante los fines de semana, cuando una parte importante de la población no trabaja ni asiste a clases.

De manera similar, la reducción de viajes entre diciembre y enero podría estar relacionada con las vacaciones y festividades de fin de año, mientras que la disminución observada entre marzo y abril podría estar asociada con el periodo vacacional de Semana Santa.

## 3. Crecimiento del sistema

Otro hallazgo interesante es el incremento notable en el número de viajes registrado durante septiembre de 2026.

Una posible explicación de este comportamiento es que el sistema MiBici haya experimentado una expansión en su número de estaciones.

Para investigar esta hipótesis, se analizó la cantidad de estaciones únicas registradas mensualmente en los datos. Los resultados muestran que, entre octubre de 2025 y mayo de 2026, el número de estaciones se mantuvo relativamente estable, alrededor de 368.

Sin embargo, a partir de junio de 2026 se observa un crecimiento considerable:

- **Mayo de 2026:** 371 estaciones.
- **Junio de 2026:** 427 estaciones.
- **Julio de 2026:** 453 estaciones.
- **Agosto de 2026:** 467 estaciones.
- **Septiembre de 2026:** 468 estaciones.

Esto representa un incremento de **97 estaciones únicas registradas entre mayo y septiembre de 2026, equivalente a un crecimiento aproximado del 26 %**.

Estos resultados respaldan la hipótesis de que el sistema experimentó una expansión durante este periodo, lo cual podría explicar, al menos parcialmente, el aumento en el número de viajes observado en septiembre.

Además, esta expansión podría contribuir a un crecimiento sostenido en el número de viajes futuros, al facilitar el acceso al sistema a una mayor cantidad de usuarios.